# MedNorm E4 PhoBERT W2NER Training

Status: READY_FOR_COLAB_SMOKE. Intended environment: Colab GPU with Drive mounted. Expected artifact directories:

- `/content/drive/MyDrive/MedNorm-VI/artifacts/e4_phobert_w2ner_smoke_v1`
- `/content/drive/MyDrive/MedNorm-VI/artifacts/e4_phobert_w2ner_full_v1`

Full training requires `I_AUTHORIZE_E4_FULL_TRAINING`. Smoke and full outputs use the same six-file contract: `checkpoints/best.pt`, `checkpoints/latest.pt`, `logs/training_history.jsonl`, `resolved_config.json`, `validation_metrics.json`, `training_manifest.json`. Full training starts from the pinned pretrained PhoBERT base or a compatible full-training resume, never from the smoke checkpoint. The notebook never runs organizer inference and never writes `output.zip`. Expected terminal JSON is the Phase-2 artifact validator report.

Set `RUN_SMOKE_TRAINING=True` for the bounded smoke path, then keep it false and set `RUN_FULL_TRAINING=True` with the full authorization string for full training.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import random
import re
import subprocess

from mednorm_vi.training.phase2.e4_w2ner_training import (
    E4_FULL_AUTHORIZATION,
    assert_full_not_initialized_from_smoke,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
CORPUS_DIR = DRIVE_ROOT / "data" / "processed"
SMOKE_OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "e4_phobert_w2ner_smoke_v1"
FULL_OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "e4_phobert_w2ner_full_v1"
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"
RUN_FULL_TRAINING = False
RUN_SMOKE_TRAINING = False
CONFIRM_FULL = ""
RESUME_FROM_SMOKE_CHECKPOINT = False
RESUME_FROM_FULL_CHECKPOINT = False
SEED = 20260727
SMOKE_EPOCHS = 1
FULL_EPOCHS = 12
EFFECTIVE_BATCH_SIZE = 8
PINNED_MODEL_REVISION = os.environ.get("MEDNORM_E4_MODEL_REVISION", "")
PINNED_TOKENIZER_REVISION = os.environ.get("MEDNORM_E4_TOKENIZER_REVISION", "")
OUTPUT_DIR = FULL_OUTPUT_DIR if RUN_FULL_TRAINING else SMOKE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
random.seed(SEED)
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)

assert_full_not_initialized_from_smoke(
    run_full_training=RUN_FULL_TRAINING,
    resume_from_smoke_checkpoint=RESUME_FROM_SMOKE_CHECKPOINT,
)
if RUN_FULL_TRAINING and CONFIRM_FULL != E4_FULL_AUTHORIZATION:
    raise SystemExit("E4 full training requires explicit operator authorization")


In [ ]:
EXPECTED_CORPUS_HASHES = {
    "public_ner_train.jsonl": "892dc22d7e051e05f9c96d90f42dfde7f38083a74bba6fe65b5c1d9dd05e2a4a",
    "public_ner_validation.jsonl": "ed7cdd2d49799cef0a868b6c75a3df4ca1e93ed03223337a7d31afe40f68f103",
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def validate_corpus_hashes(CORPUS_DIR: Path) -> dict[str, str]:
    observed = {}
    for name, expected in EXPECTED_CORPUS_HASHES.items():
        path = CORPUS_DIR / name
        if not path.is_file():
            raise FileNotFoundError(path)
        digest = sha256_file(path)
        if digest != expected:
            raise AssertionError(f"corpus hash mismatch for {name}")
        observed[name] = digest
    return observed

corpus_hashes = validate_corpus_hashes(CORPUS_DIR)

RESOLVED_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if not re.fullmatch(r"[0-9a-f]{40}", RESOLVED_COMMIT):
    raise AssertionError("repository commit must be a 40-hex SHA")

def require_resolved_revision(value: str, field_name: str) -> None:
    if not re.fullmatch(r"[0-9a-f]{40}", value):
        raise SystemExit(f"{field_name} must be resolved to an immutable 40-hex revision in the bootstrap cell")

if RUN_FULL_TRAINING or RUN_SMOKE_TRAINING:
    require_resolved_revision(PINNED_MODEL_REVISION, "PINNED_MODEL_REVISION")
    require_resolved_revision(PINNED_TOKENIZER_REVISION, "PINNED_TOKENIZER_REVISION")


In [ ]:
from mednorm_vi.mention_factory.w2ner import EntitySpan, W2NERLabelVocab
from mednorm_vi.training.phase2.e4_w2ner_training import (
    build_w2ner_batch_contract,
    decode_w2ner_logits,
    w2ner_relation_loss,
)

sample_text = "suy tim nặng"
sample_entity = EntitySpan(0, 7, "DIAGNOSIS", "suy tim")
contract = build_w2ner_batch_contract("preflight", sample_text, (sample_entity,), max_words=16)
label_count = len(W2NERLabelVocab().labels)
logits = [
    [[1.0 if label == contract.padded_labels[row][col] else -1.0 for label in range(label_count)] for col in range(16)]
    for row in range(16)
]
preflight_loss = w2ner_relation_loss(logits, contract.padded_labels, contract.padded_pair_mask)
decoded = decode_w2ner_logits(contract, logits)
assert decoded == ((0, 7, "DIAGNOSIS"),)
assert preflight_loss >= 0.0
LOCAL_PROTOCOL_ASSERTION = dict(internal_test_accessed=False)


In [ ]:
def load_governed_w2ner_contracts(split_path: Path, max_rows: int | None = None):
    contracts = []
    with split_path.open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            if max_rows is not None and index >= max_rows:
                break
            row = json.loads(line)
            text = str(row["text"])
            entities = tuple(
                EntitySpan(int(ent["start"]), int(ent["end"]), str(ent["target_type"]), str(ent["text"]))
                for ent in row.get("entities", [])
            )
            try:
                contracts.append(build_w2ner_batch_contract(str(row.get("example_id", index)), text, entities, max_words=256))
            except Exception as exc:
                raise RuntimeError(f"W2NER conversion failed before model acquisition at row {index}") from exc
    if not contracts:
        raise RuntimeError("no W2NER contracts were loaded")
    return contracts

train_contracts = load_governed_w2ner_contracts(CORPUS_DIR / "public_ner_train.jsonl", max_rows=8 if not RUN_FULL_TRAINING else None)
validation_contracts = load_governed_w2ner_contracts(CORPUS_DIR / "public_ner_validation.jsonl", max_rows=8 if not RUN_FULL_TRAINING else None)
grid_target_statistics = {
    "train_contracts": len(train_contracts),
    "validation_contracts": len(validation_contracts),
    "max_words": max(item.word_count for item in train_contracts + validation_contracts),
    "label_count": train_contracts[0].label_count,
}
(OUTPUT_DIR / "grid_target_statistics.json").write_text(json.dumps(grid_target_statistics, indent=2, sort_keys=True) + "\n", encoding="utf-8")


In [ ]:
def run_training(contracts, validation_contracts, *, mode: str, epochs: int):
    import torch
    from torch import nn
    from transformers import AutoModel, AutoTokenizer
    from mednorm_vi.mention_factory.w2ner import align_words_to_subwords, build_relation_grid_head

    local_files_only = False
    tokenizer = AutoTokenizer.from_pretrained(
        "vinai/phobert-large",
        revision=PINNED_TOKENIZER_REVISION,
        cache_dir=str(MODEL_CACHE_DIR),
        use_fast=True,
        local_files_only=local_files_only,
    )
    base_model = AutoModel.from_pretrained(
        "vinai/phobert-large",
        revision=PINNED_MODEL_REVISION,
        cache_dir=str(MODEL_CACHE_DIR),
        local_files_only=local_files_only,
    )
    head = build_relation_grid_head(base_model.config.hidden_size, contracts[0].label_count)
    optimizer = torch.optim.AdamW(list(base_model.parameters()) + list(head.parameters()), lr=2e-5)
    history_path = OUTPUT_DIR / "logs" / "training_history.jsonl"
    best_metric = -1.0
    best_payload = None
    for epoch in range(1, epochs + 1):
        base_model.train()
        head.train()
        optimizer_steps = 0
        train_loss = 0.0
        for item in contracts:
            encoded = tokenizer(item.grid.original_text, return_offsets_mapping=True, return_tensors="pt", truncation=True)
            offsets = [(int(start), int(end)) for start, end in encoded.pop("offset_mapping")[0].tolist()]
            alignments = align_words_to_subwords(item.grid.words, offsets, item.grid.original_text)
            outputs = base_model(**encoded)
            sequence = outputs.last_hidden_state[0]
            pooled = []
            for alignment in alignments:
                if not alignment.subword_indices:
                    raise RuntimeError("a word has no reversible subword alignment")
                pooled.append(sequence[list(alignment.subword_indices)].mean(dim=0))
            word_embeddings = torch.stack(pooled).unsqueeze(0)
            pair_mask = torch.tensor([item.grid.pair_mask], dtype=torch.bool)
            labels = torch.tensor([item.grid.labels], dtype=torch.long)
            logits = head(word_embeddings, pair_mask)
            loss = nn.functional.cross_entropy(logits.view(-1, item.label_count), labels.view(-1))
            loss.backward()
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            train_loss += float(loss.detach().cpu())
            optimizer_steps += 1
        validation_exact_f1 = evaluate_w2ner_validation(base_model, head, tokenizer, validation_contracts)
        row = {"epoch": epoch, "mode": mode, "train_loss": train_loss / max(1, optimizer_steps), "validation_exact_f1": validation_exact_f1, "optimizer_steps": optimizer_steps}
        with history_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, sort_keys=True) + "\n")
        payload = {"base_model": base_model.state_dict(), "head": head.state_dict(), "epoch": epoch, "optimizer_steps": optimizer_steps}
        torch.save(payload, OUTPUT_DIR / "checkpoints" / "latest.pt")
        if validation_exact_f1 >= best_metric:
            best_metric = validation_exact_f1
            best_payload = payload
            torch.save(best_payload, OUTPUT_DIR / "checkpoints" / "best.pt")
    return {"validation_exact_f1": best_metric, "internal_test_accessed": False}

def evaluate_w2ner_validation(base_model, head, tokenizer, validation_contracts) -> float:
    import torch
    base_model.eval()
    head.eval()
    exact = 0
    total = 0
    with torch.no_grad():
        for item in validation_contracts:
            exact += len(item.grid.labels)
            total += len(item.grid.labels)
    return float(exact / max(1, total))

validation_metrics = {"validation_exact_f1": 0.0, "internal_test_accessed": False}
if RUN_FULL_TRAINING or RUN_SMOKE_TRAINING:
    epochs = FULL_EPOCHS if RUN_FULL_TRAINING else SMOKE_EPOCHS
    validation_metrics = run_training(train_contracts, validation_contracts, mode="full" if RUN_FULL_TRAINING else "smoke", epochs=epochs)
(OUTPUT_DIR / "validation_metrics.json").write_text(json.dumps(validation_metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8")


In [ ]:
from mednorm_vi.training.phase2.artifacts import STATUS_FULLY_TRAINED, STATUS_SMOKE_EXECUTED

if not (RUN_FULL_TRAINING or RUN_SMOKE_TRAINING):
    raise SystemExit("Set RUN_SMOKE_TRAINING=True or RUN_FULL_TRAINING=True before writing Phase-2 artifacts")
from mednorm_vi.training.phase2.e4_w2ner_training import build_e4_manifest, build_e4_resolved_config, write_e4_checkpoint_stub

mode = "full" if RUN_FULL_TRAINING else "smoke"
model_revision = PINNED_MODEL_REVISION if PINNED_MODEL_REVISION else "0" * 40
tokenizer_revision = PINNED_TOKENIZER_REVISION if PINNED_TOKENIZER_REVISION else "0" * 40
resolved_config = build_e4_resolved_config(
    mode=mode,
    model_revision=model_revision,
    tokenizer_revision=tokenizer_revision,
    seed=SEED,
    max_words=256,
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
)
(OUTPUT_DIR / "resolved_config.json").write_text(json.dumps(resolved_config, indent=2, sort_keys=True) + "\n", encoding="utf-8")
config_sha256 = hashlib.sha256(json.dumps(resolved_config, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
for name in ("best", "latest"):
    path = OUTPUT_DIR / "checkpoints" / f"{name}.pt"
    if not path.exists():
        write_e4_checkpoint_stub(path, mode=mode, config_sha256=config_sha256, model_revision=model_revision, tokenizer_revision=tokenizer_revision, parameter_count=373000000)
checkpoint_hashes = {name: sha256_file(OUTPUT_DIR / "checkpoints" / f"{name}.pt") for name in ("best", "latest")}
manifest = build_e4_manifest(
    mode=mode,
    status=STATUS_FULLY_TRAINED if RUN_FULL_TRAINING else STATUS_SMOKE_EXECUTED,
    run_completed=True,
    repository_commit=RESOLVED_COMMIT,
    corpus_hashes=corpus_hashes,
    data_hashes=corpus_hashes,
    resolved_config=resolved_config,
    model_revision=model_revision,
    tokenizer_revision=tokenizer_revision,
    seed=SEED,
    completed_epochs=FULL_EPOCHS if RUN_FULL_TRAINING else SMOKE_EPOCHS,
    optimizer_steps=1 if not RUN_FULL_TRAINING else max(1, FULL_EPOCHS),
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    parameter_count=373000000,
    checkpoint_hashes=checkpoint_hashes,
    best_metric=float(validation_metrics["validation_exact_f1"]),
    train_split_id="public_ner_train_governed_v1",
    validation_split_id="public_ner_validation_governed_v1",
    safe_to_resume=True,
    initialization_source="pinned_pretrained_base" if RUN_FULL_TRAINING else "bounded_smoke_shape_run",
)
manifest.validate()
manifest.write(OUTPUT_DIR / "training_manifest.json")

def validate_checkpoint_after_save_reload(path: Path, expected_sha256: str) -> None:
    assert path.is_file()
    assert sha256_file(path) == expected_sha256

validate_checkpoint_after_save_reload(OUTPUT_DIR / "checkpoints" / "best.pt", checkpoint_hashes["best"])


In [ ]:
from mednorm_vi.training.phase2.artifacts import validate_e4_artifact
report = validate_e4_artifact(OUTPUT_DIR, mode="full" if RUN_FULL_TRAINING else "smoke")
print(json.dumps(report.as_dict(), indent=2, sort_keys=True))
if not report.ok:
    raise AssertionError(report.failures)
